# ROS07 · A C++ node and a custom message, built with colcon

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/03_ros2/ROS07_a_c_node_and_a_custom_message_built_with_colcon.ipynb)

**Track:** ROS 2 · **Skill axis (sheet):** Real-time and Linux · **Time:** about 75 min

**Skill Hub concepts:** rclcpp node built with colcon; Custom interfaces (msg and srv)

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("ROS07")

## Map: workspace layout and build flow

```
 ws/
 └── src/
     ├── rdl_interfaces/            interface package (ament_cmake)
     │   ├── msg/JointCommand.msg   string name / float64 position / float64 max_velocity
     │   ├── CMakeLists.txt         rosidl_generate_interfaces(...)
     │   └── package.xml            <member_of_group>rosidl_interface_packages</member_of_group>
     └── rdl_cpp/                   C++ node package (ament_cmake)
         ├── src/joint_commander.cpp
         ├── CMakeLists.txt         find_package, add_executable, target_link_libraries, install
         └── package.xml            <depend>rclcpp</depend> <depend>rdl_interfaces</depend>

 colcon build  ->  build/ (CMake trees)  install/ (the overlay)  log/
 source install/setup.bash  ->  overlay on top of the underlay (/opt/ros/jazzy or the conda env)
```

| rosidl generates from one `.msg` | Header / module |
|---|---|
| C++ | `rdl_interfaces/msg/joint_command.hpp`, type `rdl_interfaces::msg::JointCommand` |
| Python | `from rdl_interfaces.msg import JointCommand` |
| Type support for each middleware | serialization code, so every language talks the same wire format |

This lab installs compilers and colcon into the ROS environment the first time (a few extra minutes in Colab).

In [ ]:
import os
from roboprep import ros
ros.ensure_ros(extras=("build",))
for d in ["ws/src/rdl_interfaces/msg", "ws/src/rdl_cpp/src"]:
    os.makedirs(d, exist_ok=True)

## Task 1: the interface package

In [ ]:
%%writefile ws/src/rdl_interfaces/msg/JointCommand.msg
# TODO: three fields: name (string), position and max_velocity (float64)

In [ ]:
%%writefile ws/src/rdl_interfaces/CMakeLists.txt
cmake_minimum_required(VERSION 3.8)
project(rdl_interfaces)
find_package(ament_cmake REQUIRED)
find_package(rosidl_default_generators REQUIRED)
# TODO: generate code for msg/JointCommand.msg
ament_export_dependencies(rosidl_default_runtime)
ament_package()

In [ ]:
%%writefile ws/src/rdl_interfaces/package.xml
<?xml version="1.0"?>
<package format="3">
  <name>rdl_interfaces</name>
  <version>0.1.0</version>
  <description>Messages for the robotics daily labs</description>
  <maintainer email="you@example.com">BZ</maintainer>
  <license>GPL-3.0-only</license>
  <buildtool_depend>ament_cmake</buildtool_depend>
  <buildtool_depend>rosidl_default_generators</buildtool_depend>
  <exec_depend>rosidl_default_runtime</exec_depend>
  <!-- TODO: declare this package as an interface package -->
  <export><build_type>ament_cmake</build_type></export>
</package>

## Task 2: the C++ node
Node `joint_commander` publishes `rdl_interfaces/msg/JointCommand` on `joint_command` at 10 Hz:
name `"shoulder"`, position = 0.01 x (count of messages sent so far), max_velocity 1.5.

In [ ]:
%%writefile ws/src/rdl_cpp/src/joint_commander.cpp
#include <chrono>
#include <memory>
#include "rclcpp/rclcpp.hpp"
#include "rdl_interfaces/msg/joint_command.hpp"
using namespace std::chrono_literals;

class JointCommander : public rclcpp::Node {
public:
  JointCommander() : Node("joint_commander") {
    // TODO: create the publisher and a 100 ms wall timer whose lambda fills and publishes a message
  }

private:
  rclcpp::Publisher<rdl_interfaces::msg::JointCommand>::SharedPtr pub_;
  rclcpp::TimerBase::SharedPtr timer_;
  int count_ = 0;
};

int main(int argc, char **argv) {
  rclcpp::init(argc, argv);
  rclcpp::spin(std::make_shared<JointCommander>());
  rclcpp::shutdown();
  return 0;
}

In [ ]:
%%writefile ws/src/rdl_cpp/CMakeLists.txt
cmake_minimum_required(VERSION 3.8)
project(rdl_cpp)
find_package(ament_cmake REQUIRED)
find_package(rclcpp REQUIRED)
find_package(rdl_interfaces REQUIRED)
# TODO: add_executable, target_link_libraries (rclcpp::rclcpp and ${rdl_interfaces_TARGETS}),
#       install(TARGETS ... DESTINATION lib/${PROJECT_NAME}) so `ros2 run` can find it
ament_package()

In [ ]:
%%writefile ws/src/rdl_cpp/package.xml
<?xml version="1.0"?>
<package format="3">
  <name>rdl_cpp</name>
  <version>0.1.0</version>
  <description>C++ node for the robotics daily labs</description>
  <maintainer email="you@example.com">BZ</maintainer>
  <license>GPL-3.0-only</license>
  <buildtool_depend>ament_cmake</buildtool_depend>
  <depend>rclcpp</depend>
  <depend>rdl_interfaces</depend>
  <export><build_type>ament_cmake</build_type></export>
</package>

Build (about a minute), then inspect.

In [ ]:
ros.ros("cd ws && colcon build --event-handlers console_cohesion+ 2>&1 | tail -15", timeout=900)
ros.ros("source ws/install/setup.bash && ros2 interface show rdl_interfaces/msg/JointCommand")

In [ ]:
lab.check("build_run")

## Checkpoint: Math
**M1 (Apply).** The `std::chrono` literal for a 10 Hz timer, in milliseconds?

**M2 (Apply).** A 6-joint arm sends one `JointCommand` per joint at 500 Hz. Messages per second?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Evaluate).** Why put `JointCommand.msg` in its own package instead of inside `rdl_cpp`?
- a) colcon requires it
- b) any package in any language can depend on the interfaces without pulling in the node code, and messages change rarely while nodes change often
- c) messages build faster that way

**K2 (Understand).** Explain underlay vs overlay, what `source install/setup.bash` changes, and why a new terminal
cannot find `rdl_cpp` until you source it.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `ros07.h`).
3. In `roboprep/checks/ros07.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()